<a href="https://colab.research.google.com/github/Ahiyawesome/FLCBootcampTahmid2025/blob/main/education_eval_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.8/55.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 145.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.2/421.2 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 126.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 133.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22

In [3]:
# sampled from https://huggingface.co/datasets/unsloth/notebooks/blob/main/Alpaca_%2B_Llama_7b_full_example.ipynb

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch

max_seq_length = 2048 # Can go up to 8192 if needed
dtype = None          # Auto-detects T4 GPU settings


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3", # Used Llama-3.2-3B-Instruct, Qwen2.5-7B-Instruct, Llama-3.1-8B-Instruct, mistral-7b-instruct-v0.3
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = True,
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "mistral", # used llama-3.1, chatml, mistral
)

# Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)


from datasets import load_dataset

dataset = load_dataset("json", data_files="better_prompt_dialogue_train_B.jsonl", split="train")

# alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

# ### Instruction:
# {}

# ### Input:
# {}

# ### Response:
# {}"""



# llama3_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

# {instruction}<|eot_id|><|start_header_id|>user<|end_header_id|>

# {input}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

# {output}<|eot_id|>"""


# qwen_prompt = """<|im_start|>system
# {instruction}<|im_end|>
# <|im_start|>user
# {input}<|im_end|>
# <|im_start|>assistant
# {output}<|im_end|>"""

mistral_prompt = """<s>[INST] {instruction}

{input} [/INST] {output}</s>"""


def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # text = llama3_prompt.format (
        #     instruction=instruction,
        #     input=input_text,
        #     output=output
        #   )

        # text = qwen_prompt.format (
        #     instruction=instruction,
        #     input=input_text,
        #     output=output
        #   )

        text = mistral_prompt.format (
          instruction=instruction,
          input=input_text,
          output=output
        )
        texts.append(text)
    return { "text" : texts, }

# Apply the mapping
dataset = dataset.map(formatting_prompts_func, batched = True)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.6: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

Unsloth 2026.4.6 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1322 [00:00<?, ? examples/s]

In [4]:
print(dataset[2]["text"])

<s>[INST] The student's last utterance contains a mistake. The AI tutor responds to this mistake. Your task is to assess whether the tutor's response successfully identifies the mistake made by the student. The following is a tutoring dialogue in the domain of mathematics. Based on the conversation history above, your task is to evaluate the following Tutor's Response and determine whether it successfully identifies the error in the student's reasoning. Answer with just the label: Yes, No, or To some extent.

Dialogue:
Tutor: Hi, could you please provide a step-by-step solution for the question below? The question is: Marie, the confectioner, makes 12 chocolate eggs, each weighing 10 ounces. She then packs an equal number of the eggs in 4 different gift boxes. Unfortunately, she leaves one of the boxes by the kitchen window and the afternoon sun melts everything. She tosses that box out. What is the total weight, in ounces, of the remaining chocolate eggs? 
 Student: Marie made 12 choc

In [5]:
import gc
import torch
import shutil
import os
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# --- CONFIG ---
max_seq_length = 2048

# --- TRAIN ---
print("Starting Training...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8, # Used 4 and 8
        warmup_steps = 5,
        num_train_epochs = 3, # BJTU used epochs instead of max_steps, used 3 and 5
        # max_steps = 300, # Used steps 300 and 600
        learning_rate = 2e-4, #used 2e-4 for smaller models, 5e-6 for larger (actually cuz of LoRA 5e-6 doesn't work)
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

print("Saving to GGUF... this takes a few minutes.")
model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")

print("DONE! Download the model file")

Starting Training...


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1322 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,322 | Num Epochs = 3 | Total steps = 498
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.842293
2,2.109485
3,1.962868
4,1.809848
5,1.443404
6,1.349989
7,1.202002
8,0.962030
9,0.977143
10,0.886699


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


...


Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [00:51<00:00, 17.06s/it]


Unsloth: Merge process complete. Saved to `/content/model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['model_gguf/mistral-7b-instruct-v0.3.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: All GGUF conversions completed successfully!
Generated files: ['model_gguf/mistral-7b-instruct-v0.3.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model model_gguf/mistral-7b-instruct-v0.3.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to model_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f model_gguf/Modelfile
DONE! Download the model file


In [6]:
from google.colab import drive
import shutil
import glob
import os

# Mount Google Drive
drive.mount('/content/drive')

# Find the gguf file. Unsloth usually saves it locally based on the name provided.
gguf_files = glob.glob('*.gguf') + glob.glob('model*/**/*.gguf', recursive=True)

if gguf_files:
    for file_path in gguf_files:
        file_name = os.path.basename(file_path)
        dest_path = f'/content/drive/MyDrive/{file_name}'
        print(f"Copying {file_path} to Google Drive ({dest_path})... This may take a few minutes.")
        shutil.copyfile(file_path, dest_path)
        print(f"Successfully copied {file_name} to your Google Drive!")
else:
    print("Could not find the GGUF file. Please check the exact folder path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copying model_gguf/mistral-7b-instruct-v0.3.Q4_K_M.gguf to Google Drive (/content/drive/MyDrive/mistral-7b-instruct-v0.3.Q4_K_M.gguf)... This may take a few minutes.
Successfully copied mistral-7b-instruct-v0.3.Q4_K_M.gguf to your Google Drive!


In [ ]:
from google.colab import files


# file_path = '/content/model_gguf/llama-3.1-8b-instruct.Q4_K_M.gguf'
# file_path = '/content/model_gguf/qwen2.5-7b-instruct.Q4_K_M.gguf'
file_path = '/content/model_gguf/mistral-7b-instruct-v0.3.Q4_K_M.gguf'

files.download(file_path)

In [8]:
from google.colab import runtime
runtime.unassign()